In [1]:
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

get_ipython().run_line_magic('load_ext', 'sparksql_magic')
get_ipython().run_line_magic('config', 'SparkSql.limit=20')

builder = (SparkSession.builder
           .appName("delta-idempotency")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder, ['org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1']).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a9e357df-4669-4994-bb69-ee245e11f42a;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.1 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in centra

In [3]:
%%sparksql
CREATE OR REPLACE TABLE default.users (
    id INT,
    name STRING,
    age INT,
    gender STRING,
    country STRING
) USING DELTA LOCATION '/opt/workspace/data/delta_lake/idempotent-stream-write-delta/users';

In [4]:
df = (spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "kafka:9092")
      .option("subscribe", "users")
      .option("startingOffsets", "earliest")
      .load())

In [5]:
schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('age', IntegerType(), True),
    StructField('gender', StringType(), True),
    StructField('country', StringType(), True)])

df = df.withColumn('value', from_json(col('value').cast("STRING"), schema))

In [6]:
df = df.select(
    col('value.id').alias('id'),
    col('value.name').alias('name'),
    col('value.age').alias('age'),
    col('value.gender').alias('gender'),
    col('value.country').alias('country'))

In [7]:
query = (df.writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", "/opt/workspace/data/delta_lake/idempotent-stream-write-delta/users/_checkpoints/")
         .start("/opt/workspace/data/delta_lake/idempotent-stream-write-delta/users"))

25/06/10 10:18:28 ERROR TaskSetManager: Task 0 in stage 59.0 failed 4 times; aborting job
25/06/10 10:20:08 ERROR TaskSetManager: Task 0 in stage 112.0 failed 4 times; aborting job
25/06/10 10:21:48 ERROR TaskSetManager: Task 0 in stage 165.0 failed 4 times; aborting job
25/06/10 10:23:29 ERROR TaskSetManager: Task 0 in stage 218.0 failed 4 times; aborting job
25/06/10 10:25:09 ERROR TaskSetManager: Task 0 in stage 271.0 failed 4 times; aborting job
25/06/10 10:26:50 ERROR TaskSetManager: Task 0 in stage 324.0 failed 4 times; aborting job


In [11]:
app_id = 'idempotency'
def writeToDeltaLakeTableIdempotent(batch_df, batch_id):
    (batch_df.filter("country in ('Moldova', 'Germany', 'UK')")
     .write
     .format("delta")
     .mode("append")
     .option("txnVersion", batch_id)
     .option("txnAppId", app_id)
     .save("/opt/workspace/data/delta_lake/idempotent-stream-write-delta/user_europe"))
    (batch_df.filter("country IN ('USA', 'Canada', 'Brazil')")
     .write
     .format("delta")
     .mode("append")
     .option("txnVersion", batch_id)
     .option("txnAppId", app_id)
     .save("/opt/workspace/data/delta_lake/idempotent-stream-write-delta/user_americas"))
    (batch_df.filter("country IN ('India', 'China')")
     .write
     .format("delta")
     .mode("append")
     .option("txnVersion", batch_id)
     .option("txnAppId", app_id)
     .save("/opt/workspace/data/delta_lake/idempotent-stream-write-delta/user_asia"))
    (batch_df.filter("country IN ('Australia')")
     .write
     .format("delta")
     .mode("append")
     .option("txnVersion", batch_id)
     .option("txnAppId", app_id)
     .save("/opt/workspace/data/delta_lake/idempotent-stream-write-delta/user_australia"))


In [12]:
write_query = (df
               .writeStream
               .format("delta")
               .queryName("Users By Region")
               .foreachBatch(writeToDeltaLakeTableIdempotent)
               .start())

25/06/10 10:31:53 ERROR TaskSetManager: Task 0 in stage 508.0 failed 4 times; aborting job
25/06/10 10:33:22 ERROR TaskSetManager: Task 0 in stage 736.0 failed 4 times; aborting job
25/06/10 10:33:25 ERROR TaskSetManager: Task 0 in stage 744.0 failed 4 times; aborting job
25/06/10 10:33:28 ERROR TaskSetManager: Task 0 in stage 753.0 failed 4 times; aborting job
25/06/10 10:33:33 ERROR TaskSetManager: Task 0 in stage 765.0 failed 4 times; aborting job
25/06/10 10:33:34 ERROR TaskSetManager: Task 0 in stage 768.0 failed 4 times; aborting job


In [13]:
%%sparksql
SELECT COUNT(*) FROM delta.`/opt/workspace/data/delta_lake/idempotent-stream-write-delta/user_europe`;

count(1)
117


In [ ]:
%%sparksql
SELECT COUNT(*) FROM delta.`/opt/workspace/data/delta_lake/idempotent-stream-write-delta/user_aus`;